analysis.ipynb
aufruf mit notebooks/analysis.ipynb

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Design-Einstellungen
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Verbindung zur DB
conn = sqlite3.connect("../data/fitness_mock.db")

# Erster Check: Die 'trainings'-Tabelle laden
df_trainings = pd.read_sql_query("SELECT * FROM trainings", conn)
df_trainings['datum'] = pd.to_datetime(df_trainings['datum'])

df_trainings.head()

In [ ]:
# --- 1. Visualisierung der Trainingsprogression ---

# 1. Stil-Definition für Publikationen
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# 2. Erstellung des Plots

colors = {"Krafttraining": "#2c3e50", "Dauerlauf": "#27ae60", "Sprint": "#e74c3c"}
types = df_trainings['typ'].unique()

for t in types:
    subset = df_trainings[df_trainings['typ'] == t]
    plt.scatter(
        subset['datum'],
        subset['wertung'],
        label=t,
        color=colors.get(t, "blue"),
        alpha=0.6,
        s=50,
        edgecolors='w', # Weiße Umrandung für bessere Trennung bei Überlappung
        linewidth=0.5
    )

# 3. Beschriftung und Formatierung
plt.title('Analyse der Leistungsentwicklung nach Trainingstyp', fontsize=16, pad=20)
plt.xlabel('Datum', fontsize=12)
plt.ylabel('Wertung', fontsize=12)

plt.xticks(rotation=45)

# Legende außerhalb des Grids platzieren für bessere Lesbarkeit
plt.legend(title="Art des Trainings", bbox_to_anchor=(1.05, 1), loc='upper left')

# 4. Finales Layout-Adjustment
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Explorative Datenanalyse: Ausreißer-Erkennung via Boxplots ---

# 1. Datenbasis für beide Analysen direkt aus der DB laden
query_kraft = """
SELECT t.datum, k.name, (k.saetze * k.wdh * k.gewicht) as volumen
FROM trainings t
JOIN krafttraining_uebungen k ON t.id = k.training_id
WHERE t.typ = 'Krafttraining'
"""
df_kraft = pd.read_sql_query(query_kraft, conn)
# Filter auf Kniebeugen für den Boxplot
df_squats_raw = df_kraft[df_kraft["name"] == "Kniebeugen"]

query_lauf = "SELECT distanz_km FROM dauerlauf_details"
df_lauf = pd.read_sql_query(query_lauf, conn)


# 2. Erstellung von zwei Subplots nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot für Krafttraining (Kniebeugen Gesamtvolumen - noch unbereinigt)
sns.boxplot(
    data=df_squats_raw,
    x="volumen",
    ax=axes[0],
    color="#2c3e50",
    flierprops=dict(markerfacecolor="r", marker="D"),
)
axes[0].set_title(
    "Volumenverteilung Kniebeugen\n",
    fontsize=12,
    pad=10,
)
axes[0].set_xlabel("Gesamtvolumen (kg)")

# Boxplot für Dauerlauf (Distanzen)
sns.boxplot(
    data=df_lauf,
    x="distanz_km",
    ax=axes[1],
    color="#27ae60",
    flierprops=dict(markerfacecolor="r", marker="D"),
)
axes[1].set_title(
    "Distanzverteilung Dauerlauf\n",
    fontsize=12,
    pad=10,
)
axes[1].set_xlabel("Distanz (km)")

plt.suptitle(
    "Explorative Datenanalyse: Erkennung von Anomalien",
    fontsize=14,
    weight="bold",
    y=1.05,
)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Progressionsanalyse mit statistischer Datenbereinigung ---

# 1. Datenbasis laden und sortieren
df_squats = (
    df_kraft[df_kraft["name"] == "Kniebeugen"].sort_values("datum").copy()
)

# 2. Statistisches Data Cleaning
mean_vol = df_squats["volumen"].mean()
std_vol = df_squats["volumen"].std()

# Filter, um Ausreißer zu bereinigen
df_squats_cleaned = df_squats[
    (df_squats["volumen"] >= mean_vol - 2 * std_vol)
    & (df_squats["volumen"] <= mean_vol + 2 * std_vol)
].copy()

# 3. Rolling Average
df_squats_cleaned["volumen_trend"] = (
    df_squats_cleaned["volumen"].rolling(window=5, min_periods=1).mean()
)

# EINHEITLICHKEIT: Theme explizit setzen vor Figure-Erstellung
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))  # Gleiche Dimension wie Plot 1 für Symmetrie

# 4. Plot der gereinigten Daten
sns.scatterplot(
    data=df_squats_cleaned,
    x="datum",
    y="volumen",
    color="#2c3e50",
    alpha=0.6,
    s=50,
    label="Bereinigte Rohdaten",
)

sns.lineplot(
    data=df_squats_cleaned,
    x="datum",
    y="volumen_trend",
    color="#e74c3c",
    linewidth=2.5,
    label="Progressionstrend",
)

# EINHEITLICHKEIT: Titel-Stil und Schriftgrößen angepasst
plt.title(
    "Progressionsanalyse: Kniebeugen (Gesamtvolumen vs. Trend)",
    fontsize=16,
    pad=20,
)
plt.xlabel("Datum", fontsize=12)
plt.ylabel("Volumen (kg)", fontsize=12)

# Erzwingt die automatische, saubere Skalierung der Datums-Ticks wie in Schritt 1
plt.gca().xaxis.set_major_locator(plt.MaxNLocator(nbins=10))
plt.xticks(rotation=45)
# -----------------------------------

plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Feature-Korrelation (Heatmap) ---

# 1. Datenbasis vorbereiten (Nutzt die bereinigten Trainingsdaten aus der Gesamttabelle)
# Wir filtern extreme Ausreißer bei der Dauer heraus, um die Statistik nicht zu verzerren
df_trainings_cleaned = df_trainings[df_trainings["dauer_min"] < 120].copy()

# Da korrumpierte Krafteinheiten eine Wertung von 0 haben, filtern wir diese ebenfalls aus
df_trainings_cleaned = df_trainings_cleaned[
    df_trainings_cleaned["wertung"] > 0
]

# 2. Auswahl der numerischen Kernvariablen für die Matrix
all_features = df_trainings_cleaned[["dauer_min", "wertung"]]
corr_matrix = all_features.corr()

plt.figure(figsize=(6, 4))

# 3. Erstellung der Heatmap
sns.heatmap(
    corr_matrix,
    annot=True,  # Schreibt die exakten Korrelationswerte in die Quadrate
    cmap="coolwarm",  # Blau = negativ, Weiß = neutral, Rot = positiv
    vmin=-1,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Korrelationsfaktor (r)"},
)

plt.title(
    "Zusammenhangsanalyse: Trainingsdauer vs. Wertung", fontsize=14, pad=15
)
plt.tight_layout()
plt.show()